# Objašnjenje bolesti na osnovu simptoma

Notebook demonstrira generisanje objašnjenja o bolesti zasnovanoj na rezultatima prethodnog sloja - **neo4j inference**.

In [ ]:
%pip install groq python-dotenv

## LLM model

U okviru implementiranog sistema za podršku kliničkom odlučivanju, kao centralna komponenta za rezonovanje i generisanje prirodnog jezika korišćen je model **GPT-OSS 120B**. Njegove ključne karakteristike su:

- **Arhitektura mešavine eksperata** (*Mixture-of-Experts - MoE*): Model se zasniva na arhitekturi od 120 milijardi parametara, gde se procesiranje vrši putem specijalizovanih neuronskih pod-mreža ("eksperata"). Ovakva struktura omogućava modelu da aktivira najrelevantnije puteve za specifičan domen (u ovom slučaju medicinski), obezbeđujući duboku kontekstualnu analizu uz visoku računarsku efikasnost.

- **Optimizacija za logičko rezonovanje** (*Reasoning-tuned*): Model je posebno podešen (fine-tuned) za zadatke koji zahtevaju "Chain-of-Thought" (lanac razmišljanja). Ova karakteristika je ključna za interpretaciju rezultata dobijenih iz graf baze podataka (Neo4j), jer omogućava logičko povezivanje prisutnih i odsustvujućih kliničkih znakova.

- **Strogo pridržavanje strukture izlaza**: Poseduje visoku sposobnost generisanja validnog JSON formata, što je neophodno za integraciju LLM-a sa ostatkom softverskog rešenja. Model precizno poštuje definisane šeme podataka.

- **Višejezička klinička kompetencija**: Model pokazuje visok stepen morfološke i sintaksne ispravnosti pri generisanju stručnog teksta na srpskom jeziku. Sposoban je da adaptira medicinsku terminologiju u skladu sa standardima, izbegavajući kolokvijalne izraze.

In [2]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("LLM_API_KEY")

client = Groq(api_key=groq_api_key)

MODEL = "openai/gpt-oss-120b"

## Sistemski prompt

Prompt je dizajniran da transformiše sirove podatke iz sloja za zaključivanje i bodovanje u strukturiran, medicinski relevantan izveštaj. On primorava model da deluje kao **"objašnjiva AI" (XAI) komponenta**, koja ne donosi samostalne odluke, već objašnjava dobijene rezultate.

Ključni delovi prompt-a:

1. **Deterministička taksonomija nivoa poverenja**

    Sistem eliminiše subjektivnu procenu modela uvođenjem fiksnih pragova za evaluaciju pouzdanosti. Nivo poverenja (confidence score) se računa isključivo iz metrika pokrivenosti simptoma (disease_coverage i input_coverage). Ovakva formalizacija osigurava ponovljivost (reliability) sistema, gde identični ulazni parametri konzistentno generišu isti nivo dijagnostičke sigurnosti.

2. **Semantička pozadina prilikom objašnjena**

    Prompt eksplicitno zabranjuje modelu korišćenje eksternog znanja stečenog tokom pre-treninga, ograničavajući ga na eksplicitne skupove podataka iz baze znanja (Neo4j). Time se minimizuje rizik od fenomena halucinacija i osigurava da se svaka tvrdnja u obrazloženju može pratiti do izvora.

3. **Lingvistička lokalizacija**

    Prompt definiše stroga pravila za generisanje izlaza na srpskom jeziku uz poštovanje kliničkog registra. Fokus je na upotrebi standardizovane medicinske terminologije. Cilj je postizanje visokog stepena prirodnosti i profesionalizma u komunikaciji sa krajnjim korisnikom (lekarom).

4. **Strukturna validnost**

    Izlaz je striktno definisan u JSON formatu bez tekstualnih dodataka. Format izlaza:
    - `most_likely`: Ime bolesti sa najboljim poklapanjem na osnovu unetih simptoma
    - `confidence`: Stepen pouzdanosti zasnovan na skoru i coverage metrikama
      (visoka | umerena | niska)
    - `differentials`: Lista alternativnih bolesti koje treba razmotriti —
      bolesti sa sličnim simptomima ili bliskim skorom, uz uslov da im
      disease_coverage nije ispod 40%
    - `reasoning`: Kliničko obrazloženje zasnovano isključivo na ulaznim
      podacima — zašto data bolest ima najviši skor, analiza matched symptoms
      i coverage metrika. Sve što nije potkrepljeno ulaznim podacima mora biti
      označeno sa "Napomena: ovo nije potkrepljeno ulaznim podacima."
    - `recommendation`: Konkretne preporuke za lekara — koji specijalista
      da se konsultuje, koji laboratorijski testovi da se urade i
      značajni simptomi koji nisu navedeni a treba ih proveriti

## Učitavanje prompt-a

In [3]:
from pathlib import Path

BASE_DIR = Path.cwd()
prompt_path = BASE_DIR / "tests" / "xai-llm-test" / "prompts" / "disease-explanation-prompt-2"

REASONING_PROMPT = prompt_path.read_text(encoding="utf-8")

## Metoda za formatiranje ulaza u LLM model

In [4]:
def format_reasoning_input(disease_results: list[dict]) -> str:
    lines = []
    for r in disease_results:
        missing = r.get("missing_symptoms", [])
        missing_str = ", ".join(missing) if missing else "none"
        lines.append(
            f"- {r['disease']} ("
            f"uri: {r.get('uri', 'N/A')}, "
            f"score: {r['score']}, "
            f"matched: {r['match']} symptoms, "
            f"disease_coverage: {r.get('disease_coverage', 'N/A')}, "
            f"input_coverage: {r.get('input_coverage', 'N/A')}, "
            f"matched_symptoms: {r['matched_symptoms']}, "
            f"missing_symptoms: [{missing_str}])"
        )
    return "\n".join(lines)

## Podešavanje parametara LLM modela

In [18]:
import re
import time
import json

def reasoning_layer(disease_results: list[dict], max_retries: int = 3) -> dict:
    formatted_input = format_reasoning_input(disease_results)

    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": REASONING_PROMPT},
                    {"role": "user", "content": formatted_input}
                ],
                temperature=0.1,
                top_p=1.0,
                max_tokens=2500,
                stream=False,
                response_format={"type": "json_object"},
            )

            raw = completion.choices[0].message.content

            try:
                result = json.loads(raw)
                return result
            except json.JSONDecodeError:
                pass

            match = re.search(r'\{.*\}', raw, re.DOTALL)
            if match:
                result = json.loads(match.group())
                return result

            print(f"Pokusaj {attempt + 1}/{max_retries} nije vratio validan JSON. Pokusavam ponovo...")
            time.sleep(1)

        except Exception as e:
            print(f"Greška na pokušaju {attempt + 1}/{max_retries}: {e}")
            time.sleep(1)
            
    print("Svi pokusaji su propali.")
    return {
        "most_likely": "unknown",
        "confidence": "niska",
        "differentials": [],
        "reasoning": "Nije moguce generisati obrazlozenje.",
        "recommendation": "Konsultujte lekara."
    }

In [19]:
from pathlib import Path
import json

BASE_DIR = Path.cwd()

test_cases_path = BASE_DIR / "tests" / "xai-llm-test" / "test-1.json"

with open(test_cases_path, "r", encoding="utf-8") as f:
    test_case_1 = json.load(f)

print(f"Uspešno učitani podaci za prvi test")

Uspešno učitani podaci za prvi test


In [20]:
result = reasoning_layer(test_case_1)

print("Most likely:", result["most_likely"])
print("Confidence:", result["confidence"])
print("Differentials:", result["differentials"])
print("Reasoning:", result["reasoning"])
print("Recommendation:", result["recommendation"])

Most likely: hepatitis D
Confidence: niska
Differentials: ['hepatitis E']
Reasoning: Pacijent je prijavio letargiju, žutu boju kože i tamni urin, što se podudara sa tri ključna simptoma hepatitis D. Međutim, od tipičnih manifestacija hepatitis D nedostaju važni znakovi kao što su umor, anoreksija, mučnina, upala jetre (povišene transaminaze) i acholični stol, koji se nalaze u missing_symptoms. Nedostatak neuroloških simptoma (zaspalost, konfuzija, koma) i abnormalnog ponašanja dodatno smanjuje verovatnost potpune kliničke slike. Iako je input_coverage 100%, disease_coverage je samo 23,1%, što ukazuje na slab pokrivenost poznatih simptoma bolesti. Stoga je predložena verovatnoća niska i potrebno je dodatno ispitivanje.
Recommendation: Preporučuje se konsultacija hepatologa i kompletna laboratorijska evaluacija (HBV serologija, anti‑HDV antitela, PCR za HDV, testovi jetrene funkcije, koagulacija). Takođe je potrebno klinički proveriti umor, anoreksiju, mučninu i prisustvo acholičnog stol

Most likely: hepatitis D

Confidence: niska

Differentials: ['hepatitis E']

Reasoning: Pacijent je prijavio letargiju, žutu boju kože i tamni urin, što su tri simptoma koji se podudaraju sa profilom hepatitis D (uri: http://purl.obolibrary.org/obo/DOID_2047). Međutim, od tipičnih simptoma hepatitis D nedostaju ključni znakovi poput umora, anoreksije, mučnine, zapaljenja jetre i promena stolice (acholic stool), što smanjuje pokrivenost bolesti na 23,1%. Iako su svi prijavljeni simptomi pokriveni (input_coverage 100%), nedostatak većine karakterističnih manifestacija ukazuje na nisku verovatnoću. Hepatitis E (uri: http://purl.obolibrary.org/obo/DOID_4411) takođe deli žutu boju i tamni urin, ali ima manju ocenu i pokrivenost (18,2%). Stoga je hepatitis D najverovatniji, ali s velikom nesigurnošću i potrebno je razmotriti i hepatitis E.

Recommendation: Preporučuje se konsultacija hepatologa i kompletna laboratorijska evaluacija jetre (ALT, AST, bilirubin, koagulacijski parametri) uz serološke testove za hepatitis D i E. Takođe je indicirana ultrazvučna kontrola jetre. Lekaru se savetuje da proveri odsutne, ali klinički značajne simptome: umor, anoreksiju, mučninu, zapaljenje jetre (bol u desnom podrebrju, povišeni enzimi) i acholic stool. Ako postoje faktori rizika (putovanja, izloženost HBV), to treba uzeti u obzir pri daljoj dijagnostici.

In [8]:
from pathlib import Path
import json

BASE_DIR = Path.cwd()

test_cases_path = BASE_DIR / "tests" / "xai-llm-test" / "test-2.json"

with open(test_cases_path, "r", encoding="utf-8") as f:
    test_case_2 = json.load(f)

print(f"Uspešno učitani podaci za drugi test")

Uspešno učitani podaci za drugi test


In [9]:
result = reasoning_layer(test_case_2)

print("Most likely:", result["most_likely"])
print("Confidence:", result["confidence"])
print("Differentials:", result["differentials"])
print("Reasoning:", result["reasoning"])
print("Recommendation:", result["recommendation"])

Most likely: 3-methylcrotonyl-CoA carboxylase deficiency
Confidence: visoka
Differentials: ['beta-ketothiolase deficiency', 'swine influenza']
Reasoning: Pacijent ima dijareju, letargiju i povraćanje, što se podudara sa tri od četiri tipična simptoma 3‑metilkrutonil‑CoA karboksilaze deficijencije. Nedostaje jedini ključni simptom iz profila ove bolesti – loša ishrana, koji se često javlja kod novorođenčadi i male dece. Beta‑ketotiozolski deficit pokazuje samo letargiju i povraćanje, a nedostaju mu karakteristični napadi i dehidracija, što ga čini manje verovatnim. Svinska influenza takođe obuhvata dijareju, letargiju i povraćanje, ali njen profil zahteva respiratorne simptome (kašalj, groznicu, rinoreju) koji nisu izvezeni kod pacijenta. Stoga je najverovatnija metabolička bolest, dok su druge dijagnoze moguće, ali manje usklađene sa celokupnim simptomatskim spektrom.
Recommendation: Preporučuje se hitna konsultacija pedijatrijskog metaboličkog/genetskog specijaliste i laboratorijska i

Most likely: 3-methylcrotonyl-CoA carboxylase deficiency

Confidence: visoka

Differentials: ['beta-ketothiolase deficiency', 'swine influenza']

Reasoning: Pacijentova klinička slika uključuje dijareju, letargiju i povraćanje, što se poklapa sa tri od četiri tipična simptoma 3-methylcrotonyl-CoA karboksilaza deficijencije. Nedostaje jedini ključni simptom iz profila bolesti – loše hranjenje, što može biti prisutno u ranim fazama i treba ga proveriti. Beta‑ketotiozola deficijencija deli letargiju i povraćanje, ali obično se javlja i napadi i dehidracija, koji nisu izvezeni kod pacijenta. Svinja grip (influenza) takođe može izazvati dijareju, letargiju i povraćanje, ali tipično je praćena respiratornim simptomima (kašalj, groznica, rinoreja) koji nisu prijavljeni. Stoga, iako su simptomi preklapajući, nedostatak respiratornih znakova i prisustvo gastrointestinalnih simptoma čine 3-methylcrotonyl-CoA karboksilaza deficijenciju najverovatnijom.

Recommendation: Preporučuje se hitna konsultacija sa pedijatrijskim metaboličkim/geneskim specijalistom. Laboratorijski testovi: kvantifikacija organskog kiselog profila u urinu, akilcarnitin panel i enzimatska analiza u fibroblastima ili limfocitima. Klinicki treba proveriti prisustvo lošeg hranjenja, kao i eventualne znakove hipoglikemije ili ketoacidoze. Ako se simptomi pogoršaju, razmotriti hospitalizaciju radi nadzora metabolizma.

In [10]:
from pathlib import Path
import json

BASE_DIR = Path.cwd()

test_cases_path = BASE_DIR / "tests" / "xai-llm-test" / "test-3.json"

with open(test_cases_path, "r", encoding="utf-8") as f:
    test_case_3 = json.load(f)

print(f"Uspešno učitani podaci za treći test")

Uspešno učitani podaci za treći test


In [11]:
result = reasoning_layer(test_case_3)

print("Most likely:", result["most_likely"])
print("Confidence:", result["confidence"])
print("Differentials:", result["differentials"])
print("Reasoning:", result["reasoning"])
print("Recommendation:", result["recommendation"])

Most likely: Marburg hemorrhagic fever
Confidence: visoka
Differentials: []
Reasoning: Pacijentova klinička slika sadrži dijareju, glavobolju, bol u abdomenu, upalu ždrela, makulopapularni osip, krvarenje, mučninu, zimicu i bol u grudima, što se poklapa sa većinom simptoma navedenim za Marburg virusnu bolest (DOID_4327). Nedostaju jedino povišena temperatura i povraćanje, koji su česti kod ove infekcije, ali njihova odsutnost ne isključuje dijagnozu. Poklapanje svih navedenih simptoma čini 81,8 % pokrivenosti bolesti i 100 % pokrivenosti unetih simptoma, što ukazuje na visoku verovatnoću. Iako je odsustvo groznice neobično, moguće je da se još nije razvila ili da nije prijavljena. Ostale razmatrane bolesti imaju znatno niže skorove i manju pokrivenost simptoma, te su manje verovatne.
Recommendation: Preporučuje se hitna konsultacija infektologa i izolacija pacijenta. Neophodni su laboratorijski testovi: PCR za filovirusne RNA, kompletna krvna slika, koagulacijski panel i funkcionalni t

Most likely: Marburg hemorrhagic fever

Confidence: visoka

Differentials: ['Mycoplasma pneumoniae pneumonia', 'West Nile fever']

Reasoning: Na osnovu podataka, Marburg hemoragijska groznica ima najviši skor (16.17) i pokriva 81,8 % poznatih simptoma bolesti, pri čemu su svi simptomi iz unosa pacijenta (100 % input_coverage) prisutni. Simptomi koji se podudaraju uključuju dijareju, glavobolju, bol u abdomenu, upalu ždrela, makulopapularni osip, krvarenje, mučninu, zimicu i bol u grudima. Nedostaju ključni simptomi tipični za Marburg – groznica i povraćanje, što može ukazivati na nekompletan klinički prikaz ili ranu fazu bolesti. Mycoplasma pneumoniae pneumonija i West Nile groznica imaju znatno niži skor i pokrivaju manji deo simptoma (44‑45 % input_coverage), te im nedostaju karakteristični kožni i gastrointestinalni simptomi. Stoga, iako su drugi uzroci moguće, Marburg ostaje najverovatnija dijagnoza na osnovu dostupnih podataka. Međutim, zbog ozbiljnosti i potencijalne epidemijske prirode, neophodna je hitna dodatna evaluacija.

Recommendation: Preporučuje se hitna konsultacija infektologa i izolacija pacijenta. Laboratorijski testovi: PCR za filovirus (Marburg), kompletna krvna slika, koagulacijski panel, testovi jetrene funkcije i serološki testovi. Takođe je potrebno proveriti nedostajuće kliničke znakove – groznicu, povraćanje i eventualno druge znakove krvarenja ili hemodinamske nestabilnosti. Ukoliko se pojave, treba odmah započeti odgovarajuću podršku i terapiju.

In [12]:
from pathlib import Path
import json

BASE_DIR = Path.cwd()

test_cases_path = BASE_DIR / "tests" / "xai-llm-test" / "test-4.json"

with open(test_cases_path, "r", encoding="utf-8") as f:
    test_case_4 = json.load(f)

print(f"Uspešno učitani podaci za četvrti test")

Uspešno učitani podaci za četvrti test


In [13]:
result = reasoning_layer(test_case_4)

print("Most likely:", result["most_likely"])
print("Confidence:", result["confidence"])
print("Differentials:", result["differentials"])
print("Reasoning:", result["reasoning"])
print("Recommendation:", result["recommendation"])

Most likely: swine influenza
Confidence: visoka
Differentials: ['respiratory syncytial virus infectious disease']
Reasoning: Pacijent prikazuje letargiju, kašalj, povišenu temperaturu, mučninu i rinoreju, što su ključni simptomi koji se podudaraju sa profilom svinjskog influencijskog virusa. Pokrivenost bolesti iznosi 62,5 % i sve prijavljene simptome pacijenta pokriva, što daje visoku verovatnoću. Nedostaju simptomi tipični za svinjsku gripu kao što su dijareja, upala ždrela i povraćanje, koji nisu prijavljeni kod pacijenta. Odsustvo ovih gastrointestinalnih manifestacija ne isključuje bolest, ali smanjuje potpunu kliničku slaganje. Alternativna dijagnoza, respiratorni sincicijalni virus, deli kašalj, temperaturu i rinoreju, ali pokriva manji broj simptoma (75 % disease_coverage, 60 % input_coverage) i nedostaje letargija i mučnina. Stoga je svinjska grip najverovatnija, ali je potrebno dodatno ispitivanje.
Recommendation: Preporučuje se hitna konsultacija infektologa ili pulmologa, l

Most likely: swine influenza

Confidence: visoka

Differentials: ['respiratory syncytial virus infectious disease']

Reasoning: Pacijent ima letargiju, kašalj, povišenu temperaturu, mučninu i rinitis, što se poklapa sa tipičnim kliničkim manifestacijama svinjskog influenznog virusa. Nedostaju gastrointestinalni simptomi koji su česti kod ove infekcije – dijareja, povraćanje i upala ždrela nisu prijavljeni. Pokrivenost bolesti iznosi 62,5 % i sve prijavljene simptome pacijenta pokrivaju profil bolesti (input_coverage 100 %). Ovi podaci čine svinjsku gripu najverovatnijim uzrokom u poređenju sa drugim respiratornim infekcijama. Diferencijalna dijagnoza uključuje respiratorni sincicijalni virus, koji deli kašalj, temperaturu i rinitis, ali nedostaje letargija i mučnina. Potrebno je proveriti prisustvo gastrointestinalnih simptoma i moguće izloženosti svinjama ili kontakt sa zaraženim osobama.

Recommendation: Preporučuje se hitna konsultacija infektologa ili pulmologa, laboratorijska ispitivanja uključuju PCR test za influenza A (svinjska so), brzi antigeni, kompletna krvna slika i kulturu respiratornog uzorka. Takođe treba proveriti prisustvo dijareje, upale ždrela i povraćanja. Ukoliko se potvrdi, započinje se antivirusna terapija prema smernicama. Praćenje simptoma i eventualna karantin preporučuju se.

In [14]:
from pathlib import Path
import json

BASE_DIR = Path.cwd()

test_cases_path = BASE_DIR / "tests" / "xai-llm-test" / "test-5.json"

with open(test_cases_path, "r", encoding="utf-8") as f:
    test_case_5 = json.load(f)

print(f"Uspešno učitani podaci za peti test")

Uspešno učitani podaci za peti test


In [15]:
result = reasoning_layer(test_case_5)

print("Most likely:", result["most_likely"])
print("Confidence:", result["confidence"])
print("Differentials:", result["differentials"])
print("Reasoning:", result["reasoning"])
print("Recommendation:", result["recommendation"])

Most likely: Ebola virus disease
Confidence: visoka
Differentials: []
Reasoning: Ebola virus disease pokazuje najviši skor (13.45) i pokriva 72,7 % poznatih simptoma bolesti, pri čemu su svi simptomi iz unosa pacijenta (100 % input_coverage) prisutni: dijareja, glavobolja, osip, groznica, crveno‑crvena konjunktiva, krvarenje, slabost i mišićna slabost. Nedostaju tri tipična simptoma bolesti – upala ždrela, bol u zglobovima i povraćanje – što smanjuje ukupnu pokrivenost bolesti, ali ne umanjuje značaj podudaranja. Ostale bolesti na listi imaju znatno niži skor i pokrivaju manji broj simptoma (najviše dva ili tri), što ih čini manje verovatnim. Nema kontradikcija u podudaranju simptoma; svi izvezeni simptomi su dosledni Ebola profilu. Stoga je najverovatnija dijagnoza Ebola virus disease, iako je ovo samo probabilistički predlog, a ne konačna dijagnoza.
Recommendation: Preporučuje se hitna konsultacija infektologa i/ili epidemiologa, laboratorijska ispitivanja na prisustvo Ebola virusa (

Most likely: Ebola virus disease

Confidence: visoka

Differentials: ['hymenolepiasis']

Reasoning: Ebola virus disease pokazuje najviši skor i poklapa se sa svim navedenim simptomima pacijenta: dijareja, glavobolja, osip, povišena temperatura, crvenilo očiju, krvarenje, opšta slabost i mišićna slabost. Nedostaju tri tipična simptoma za Ebola – upala ždrela, bol u zglobovima i povraćanje – što ne umanjuje značajnu pokrivenost (72,7% disease_coverage i 100% input_coverage). Ostale bolesti u listi imaju znatno manji broj podudarnih simptoma i niži ulazni pokrivenost, pa su manje verovatne. Nema logičkih kontradikcija između izveštaja simptoma i profila Ebola bolesti. Stoga je Ebola najverovatniji uzrok trenutnog kliničkog obraza.

Recommendation: Preporučuje se hitna konsultacija infektologa i izolacija pacijenta. Neophodni su laboratorijski testovi: PCR za Ebola virus, kompletna krvna slika, koagulacijski panel, testovi jetrene funkcije i serološki testovi. Takođe, lekar treba da proveri prisustvo nedostajućih simptoma – upalu ždrela, bol u zglobovima i povraćanje – i da ispita epidemiološku pozadinu (putovanja u endemicna područja, kontakt sa zaraženim osobama ili telesnim tečnostima).

In [16]:
from pathlib import Path
import json

BASE_DIR = Path.cwd()

test_cases_path = BASE_DIR / "tests" / "xai-llm-test" / "test-6.json"

with open(test_cases_path, "r", encoding="utf-8") as f:
    test_case_6 = json.load(f)

print(f"Uspešno učitani podaci za sesti test")

Uspešno učitani podaci za sesti test


In [17]:
result = reasoning_layer(test_case_6)

print("Most likely:", result["most_likely"])
print("Confidence:", result["confidence"])
print("Differentials:", result["differentials"])
print("Reasoning:", result["reasoning"])
print("Recommendation:", result["recommendation"])

Most likely: Japanese encephalitis
Confidence: umerena
Differentials: ['St. Louis encephalitis', 'Powassan encephalitis']
Reasoning: Pacijent ima glavobolju, konfuziju, komu, tremor, dezorijentaciju i ukočenost vrata, što se podudara sa kliničkim slikom japanske encefalitis. Ovi simptomi čine 60% poznatih simptoma bolesti i pokrivaju 85,7% prijavljenih simptoma, što daje umerenu verovatnoću. Nedostaju ključni simptomi koji se često javljaju kod japanske encefalitis, poput visokog groznice, konvulzija, spastične paralize i stupora. Odsustvo visokog groznice i paralize može ukazivati na drugačiji uzrok ili ranu fazu bolesti. Sličnoj kliničkoj slici odgovaraju i St. Louis encefalitis i Powassan encefalitis, ali oni imaju nešto drugačiji spektar nedostajućih simptoma, pa je potrebno dodatno ispitivanje.
Recommendation: Preporučuje se hitna konsultacija neurologije i infektologije, laboratorijska ispitivanja serološkog testa na japanski encefalitis virus, PCR u CSF, kompletna analiza cerebr

Most likely: Japanese encephalitis

Confidence: umerena

Differentials: ['St. Louis encephalitis', 'Powassan encephalitis']

Reasoning: Pacijentova klinička slika uključuje glavobolju, konfuziju, komu, tremor, dezorijentaciju i ukočenost vrata, što se podudara sa profilom japanske encefalitisa (uri: http://purl.obolibrary.org/obo/DOID_10844). Nedostaju ključni simptomi koji su tipični za ovu bolest, a to su konvulzije, visoka temperatura, spastična paraliza i stupor, što smanjuje ukupnu pokrivenost bolesti na 60 %. Slično, St. Louis encefalitis ima identičan skup podudarajućih i nedostajućih simptoma, pa je diferencijalna dijagnoza vrlo bliska. Powassan encefalitis takođe poklapa šest simptoma, ali se razlikuje po prisustvu povišene temperature i odsustvu tremora, što ga čini nešto manje verovatnim. S obzirom na visoku stopu poklapanja simptoma (85,7 % ulaznih simptoma), ali i na odsustvo ključnih znakova, predložena dijagnoza ostaje probabilistička, a ne definitivna.

Recommendation: Preporučuje se hitna konsultacija neurologa i infektologa, uz laboratorijsku evaluaciju: analiza cerebrospinalne tečnosti, PCR i serološki testovi za flavivirusne agense (uključujući japanski i St. Louis virus), kao i MRI mozga. Takođe je potrebno proveriti prisustvo sledećih nedostajućih simptoma: konvulzije, visoka temperatura, spastična paraliza i stupor, kako bi se dodatno suzio dijagnostički spektar.